In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu # importing required statistical packages

df = pd.read_excel("df_CAZy_MM_5Sep2025.xlsx", sheet_name='Sheet1') # loading the Excel sheet into a dataframe

# Define groups for comparison
group1 = "Baseline"
group2 = "TD"

In [ ]:
counts = df['TimePoints'].value_counts() 
print(counts) # checking for unexpected or misspelled TimePoints

In [ ]:
# Filter for just the two groups
df_filtered = df[df['TimePoints'].isin([group1, group2])].copy()
df_filtered['TimePoints'] = df_filtered['TimePoints'].astype('category')

Feature_cols = df.columns[20:] # selecting feature columns after the metadata columns
print(Feature_cols)  # checking that only feature columns are included


In [ ]:
results = []
p_values = []

for col in Feature_cols:
    if pd.api.types.is_numeric_dtype(df_filtered[col]):
        g1 = df_filtered.loc[df_filtered['TimePoints'] == group1, col].dropna()
        g2 = df_filtered.loc[df_filtered['TimePoints'] == group2, col].dropna()

        n1, n2 = len(g1), len(g2)
        print(f"Feature: {col} | {group1} samples: {n1} | {group2} samples: {n2}")

        if n1 > 0 and n2 > 0:
            # Mann–Whitney U test (U returned is for group1, the first argument)
            U1, p_value = mannwhitneyu(g1, g2, alternative='two-sided') # Can be either "less" or "greater" based on the hypothesis
            p_values.append(p_value)

            # Rank-biserial correlation (signed)
            rrb = (2 * U1) / (n1 * n2) - 1
            rrb_abs = abs(rrb)

            # Determine enriched cohort based on rank-biserial direction
            if rrb > 0:
                enriched_group = group1
            elif rrb < 0:
                enriched_group = group2
            else:
                enriched_group = "No difference"

            results.append({
                "Feature": col,
                "Mann-Whitney U": U1,
                "p-value": p_value,
                "RankBiserial_r": rrb,
                "RankBiserial_r_abs": rrb_abs,
                "EnrichedCohort": enriched_group
            })

In [ ]:
results_df = pd.DataFrame(results)
output_file = "mannwhitney_with_rankbiserial.xlsx"
results_df.to_excel(output_file, index=False)